# Question 4: Correlated measurements

Four measurements are recorded on each of twelve specimens. The measurements are related to
one another, so the four numbers carry less than four measurements' worth of distinct
information.

You will compute the covariance matrix, decompose it, and read off how much of the total
variance each principal direction accounts for.

Save your own copy first (File > Save a copy in Drive). Then paste the INPUTS block from the
course page over the INPUTS cell below, replacing everything in that cell.

Run the cells in order and fill in every TODO. Use only `numpy` and `matplotlib`.

In [ ]:
# ===== INPUTS =====
# Replace this whole cell with the INPUTS block from the course page.
DATA = None
THRESHOLD = 0.90
# ==================

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if DATA is None:
    raise ValueError("Paste the INPUTS block from the course page over the INPUTS cell, "
                     "run that cell, then run this one again.")

X = np.asarray(DATA, dtype=float)
n, p = X.shape
names = ["m1", "m2", "m3", "m4"]
print(f"n = {n} specimens, p = {p} measurements")
print("column means:", np.round(X.mean(axis=0), 4))

### TODO 1: the covariance matrix

Complete `covariance` so that it returns the $p \times p$ sample covariance matrix of the
rows of `X`:

$$ S \; = \; \frac{1}{n-1} X_c^{\top} X_c $$

where $X_c$ is `X` with the mean of each column subtracted.

In [ ]:
def covariance(X):
    """Sample covariance matrix of the rows of X, divisor n - 1."""
    # TODO 1: center the columns, then form the covariance matrix
    return ...

### TODO 2: the eigendecomposition, sorted

Complete `eig_sorted` so that it returns the eigenvalues and eigenvectors of the symmetric
matrix `M`, sorted so the largest eigenvalue comes first.

Use `np.linalg.eigh`, which is for symmetric matrices. It returns the eigenvalues in
*ascending* order, so you will need to reorder both its outputs with the same index array.
Column `i` of the eigenvector matrix must stay paired with eigenvalue `i`.

In [ ]:
def eig_sorted(M):
    """Eigenvalues and eigenvectors of a symmetric M, largest eigenvalue first."""
    # TODO 2: eigendecompose, then reorder both outputs, descending by eigenvalue
    return ...

### TODO 3: variance explained

Complete `variance_explained` so that it returns two arrays: the proportion of the total
variance accounted for by each component, and its running total.

$$ \mathrm{PVE}_k \; = \; \frac{\lambda_k}{\sum_{j=1}^{p} \lambda_j}
\qquad\qquad
\mathrm{CPVE}_k \; = \; \sum_{j=1}^{k} \mathrm{PVE}_j $$

`np.cumsum` gives the running total.

In [ ]:
def variance_explained(values):
    """Proportion of variance explained by each component, and the running total."""
    # TODO 3: return (pve, cumulative pve)
    return ...


S = covariance(X)
values, vectors = eig_sorted(S)
pve, cpve = variance_explained(values)

lambda_1 = values[0]
pve_1 = pve[0]
n_components = int(np.argmax(cpve >= THRESHOLD)) + 1
top_loading = np.abs(vectors[:, 0]).max()

print("covariance matrix:")
print(np.round(S, 4))
print()
print(f"{'component':>10} {'eigenvalue':>12} {'PVE':>9} {'cumulative':>12}")
for k in range(p):
    print(f"{k + 1:>10} {values[k]:>12.4f} {pve[k]:>9.4f} {cpve[k]:>12.4f}")
print()
print(f"total variance = sum of eigenvalues = {values.sum():.4f}")
print(f"              = sum of the diagonal = {np.trace(S):.4f}")
print(f"components needed to reach {THRESHOLD:.0%} of the variance: {n_components}")
print()
print("component 1 loadings:", "  ".join(f"{name} {w:+.4f}" for name, w in zip(names, vectors[:, 0])))
print(f"largest absolute loading on component 1: {top_loading:.4f}")

### See it

The bars are the proportion of variance explained by each component; the line is the running
total, with the $90\%$ mark drawn across it.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ks = np.arange(1, p + 1)
ax.bar(ks, pve, color="#9cc3e6", edgecolor="white", label="PVE per component")
ax.plot(ks, cpve, "o-", color="#B45309", label="cumulative PVE")
ax.axhline(THRESHOLD, color="red", linestyle="--", label=f"{THRESHOLD:.0%}")
ax.set(xlabel="component", ylabel="proportion of total variance",
       xticks=ks, ylim=(0, 1.05), title="Variance explained")
ax.legend()
plt.tight_layout()
plt.show()

### Report your results

Run the cell below and enter R1, R2 and R3 on the course page. R3 is an absolute value
because an eigenvector's sign is arbitrary: `v` and `-v` point along the same direction.

In [ ]:
print("=" * 60)
print("QUESTION 4: REPORT VALUES")
print("=" * 60)
print(f"R1. Largest eigenvalue (4 dec):                       {lambda_1:.4f}")
print(f"R2. PVE of the first component (4 dec):               {pve_1:.4f}")
print(f"R3. Largest absolute loading on component 1 (4 dec):  {top_loading:.4f}")

### Multiple-choice A

*This also appears on the course page, where you answer it. The options there may be in a different order.*

Look at the table of eigenvalues and cumulative PVE. Which statement is best supported?

1. Because four components are needed to reach 100% of the variance, all four measurements carry distinct information.
2. The first component accounts for the largest share of the total variance, and two components together account for more than 90% of it.
3. The first component is the measurement that varies most, so it can be read directly off the largest diagonal entry of the covariance matrix.
4. The eigenvalues sum to 1, so each one is already a proportion.

### Counterfactual

Switching `USE_CORRELATION` to `True` decomposes the correlation matrix instead of the
covariance matrix. The correlation matrix is the covariance matrix of the same data after
each measurement has been rescaled to unit variance, so measurements that happen to be
recorded on a larger scale no longer count for more.

Change `USE_CORRELATION` from `False` to `True` in the cell below, re-run that cell
only, and compare with the values you reported above.

In [ ]:
USE_CORRELATION = False     # <-- change to True and re-run this cell

if USE_CORRELATION:
    sd = np.sqrt(np.diag(S))
    M = S / np.outer(sd, sd)
else:
    M = S

values_cf, _ = eig_sorted(M)
pve_cf, cpve_cf = variance_explained(values_cf)

print(f"USE_CORRELATION = {USE_CORRELATION}")
print(f"  eigenvalues        = {np.round(values_cf, 4).tolist()}")
print(f"  they sum to          {values_cf.sum():.4f}")
print(f"  PVE of component 1 = {pve_cf[0]:.4f}")
print(f"  components for 90% = {int(np.argmax(cpve_cf >= THRESHOLD)) + 1}")

### Multiple-choice B

*This also appears on the course page, where you answer it. The options there may be in a different order.*

You set `USE_CORRELATION = True` and re-ran the cell. Which statement matches the output?

1. Every eigenvalue is unchanged, because correlation and covariance describe the same relationships.
2. The eigenvalues changed and now sum to 4, and the first component's share of the variance fell.
3. The eigenvalues changed and now sum to 4, and the first component's share of the variance rose.
4. The eigenvalues changed but still sum to the total variance of the original measurements.